# H3N2 serum split: Nextflu, Adaboost, and SCMS-FiLM-H3

This notebook automatically aggregates the 10 serum-split seeds for H3N2. It uses the `with_name` metrics for Nextflu and Adaboost, and the final-epoch test metrics for SCMS-FiLM-H3.

In [ ]:
from __future__ import annotations

import csv
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'results').is_dir() and (candidate / 'paper').is_dir():
            return candidate
    raise FileNotFoundError('Could not find the fluProfiler project root.')


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
RESULTS_ROOT = PROJECT_ROOT / 'results/H1H3_HA1_v1.0/20260717_164256'
SEEDS = range(10)
MODELS = ['SCMS-FiLM-H3', 'Nextflu', 'Adaboost']
COLORS = {'SCMS-FiLM-H3': '#C5DFF4', 'Nextflu': '#F4EEAC', 'Adaboost': '#C9DCC4'}
OUTPUT_PATH = PROJECT_ROOT / 'paper/data/Fig_H3N2_serum_model_comparison.png'


In [ ]:
def read_json_metrics(model: str, seed: int) -> dict[str, float]:
    path = RESULTS_ROOT / model / 'serum' / f'seed_{seed}' / 'H3N2' / 'metrics.json'
    with path.open(encoding='utf-8') as handle:
        metrics = json.load(handle)['metrics']
    # Nextflu stores with_name at the top level; Adaboost nests it under test/by_subtype.
    values = metrics.get('with_name')
    if values is None:
        values = metrics['test']['by_subtype']['H3N2']['with_name']
    return {'mae': float(values['mae']), 'pearson': float(values['pearson'])}


def read_scms_metrics(seed: int) -> dict[str, float]:
    path = (RESULTS_ROOT / 'SCMS-FiLM-H3-Epoch50' / 'serum' / f'seed_{seed}'
            / 'subtype/H3N2_seed42/metrics.csv')
    with path.open(newline='', encoding='utf-8') as handle:
        rows = list(csv.DictReader(handle))
    if not rows:
        raise ValueError(f'No metrics found in {path}')
    final = rows[-1]
    return {
        'mae': float(final['test_pooled_mae']),
        'pearson': float(final['test_pooled_pearson']),
    }


records = []
for seed in SEEDS:
    for model in ('Nextflu', 'Adaboost'):
        records.append({'model': model, 'seed': seed, **read_json_metrics(model, seed)})
    records.append({'model': 'SCMS-FiLM-H3', 'seed': seed, **read_scms_metrics(seed)})

per_seed = pd.DataFrame(records)
summary = (per_seed.groupby('model', sort=False)[['mae', 'pearson']]
           .agg(['mean', 'std', 'count'])
           .reindex(MODELS))
display(per_seed.sort_values(['model', 'seed']))
display(summary.round(4))


In [ ]:
mae_mean = summary[('mae', 'mean')]
mae_std = summary[('mae', 'std')]
pearson_mean = summary[('pearson', 'mean')]

fig, ax = plt.subplots(figsize=(6.2, 4.2), dpi=300)
x = range(len(MODELS))
bars = ax.bar(
    x, mae_mean, yerr=mae_std, capsize=4, width=0.65,
    color=[COLORS[model] for model in MODELS],
    error_kw={'elinewidth': 1.2, 'capthick': 1.2}, zorder=2,
)
ax.set_xticks(list(x), MODELS, rotation=15, ha='right')
ax.set_ylabel('HI titer prediction error (MAE)')
ax.set_title('H3N2: divided by serum')
ax.grid(axis='y', alpha=0.2, zorder=0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

for bar, mean, std in zip(bars, mae_mean, mae_std):
    ax.text(bar.get_x() + bar.get_width() / 2, mean + std, f'{mean:.2f}',
            ha='center', va='bottom', fontsize=9)

ax2 = ax.twinx()
ax2.plot(list(x), pearson_mean, color='black', marker='o', linewidth=1.7, markersize=5)
ax2.set_ylabel('Pearson correlation')
ax2.set_ylim(-0.2, 1.0)
ax2.spines['top'].set_visible(False)

fig.tight_layout()
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(OUTPUT_PATH, bbox_inches='tight')
print(f'Saved figure to: {OUTPUT_PATH}')
plt.show()
